# Классификация дорожных поверхностей — обучение на Colab

Этот ноутбук запускает полный цикл обучения на бесплатной GPU от Google.

## Перед запуском:

1. **GPU**: `Runtime → Change runtime type → T4 GPU`
2. **Секреты**: `Runtime → Manage secrets` → добавить:
   - `AWS_ACCESS_KEY_ID` — ключ Yandex Cloud
   - `AWS_SECRET_ACCESS_KEY` — секрет Yandex Cloud
   - `BUCKET_NAME` — имя бакета (road-surface-classification-storage-1)
   - `MLFLOW_TRACKING_URI` — URL MLflow сервера (опционально)

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/oulright/road-surface-classification.git"
BRANCH = "feature"

repo_dir = Path("/content/road-surface-classification")

if not repo_dir.exists():
    !git clone -b {BRANCH} {REPO_URL} {repo_dir}
    print(f"✓ Репозиторий склонирован в {repo_dir}")
else:
    %cd {repo_dir}
    !git pull origin {BRANCH}
    print(f"✓ Репозиторий обновлён")

%cd {repo_dir}

In [ ]:
!pip install -e ".[dev,dvc]" -q
print("✓ Зависимости установлены")

In [ ]:
!python scripts/colab_setup.py

In [ ]:
import pandas as pd

for split in ["train", "val", "test"]:
    csv_path = f"data/processed/{split}.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"\n{split}.csv: {len(df)} сэмплов")
        print(df["label"].value_counts().to_string())
    else:
        print(f"\n⚠ {csv_path} не найден")

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"Память: {mem_gb:.1f} ГБ")

## Обучение

Выберите модель:
- `simple_cnn.yaml` — быстрый baseline (~0.5M параметров)
- `resnet18_mel.yaml` — ResNet18 с предобучением (~11M параметров)
- `efficientnet_b0.yaml` — EfficientNet-B0 (~5M параметров)

In [ ]:
# НАСТРОЙКИ ОБУЧЕНИЯ

CONFIG = "configs/audio/models/simple_cnn.yaml"
EPOCHS = 50
BATCH_SIZE = 64
LR = 0.001

# ═══════════════════════════════════

!python scripts/train.py \
    --config {CONFIG} \
    --override training.epochs={EPOCHS} \
              training.batch_size={BATCH_SIZE} \
              training.lr={LR}

In [ ]:
import glob

# Находим лучший чекпоинт
checkpoints = sorted(glob.glob("checkpoints/*.pt"))
if checkpoints:
    best_ckpt = checkpoints[-1]
    print(f"Лучший чекпоинт: {best_ckpt}")

    !python scripts/evaluate.py \
        --checkpoint {best_ckpt} \
        --config {CONFIG} \
        --test-csv data/processed/test.csv \
        --save-predictions predictions.csv
else:
    print("⚠ Чекпоинтов не найдено")

In [ ]:
from google.colab import files

# Скачать предсказания
if os.path.exists("predictions.csv"):
    print("Скачиваем predictions.csv...")
    files.download("predictions.csv")

# Скачать лучший чекпоинт
checkpoints = sorted(glob.glob("checkpoints/*.pt"))
if checkpoints:
    best = checkpoints[-1]
    print(f"Скачиваем {best}...")
    files.download(best)

## Сохранение на Google Drive (опционально)

Раскомментируйте ячейку ниже, чтобы сохранить результаты на Google Drive.
Так вы не потеряете результаты при отключении Colab.

In [ ]:
# # Раскомментируйте для сохранения на Google Drive:
#
# from google.colab import drive
# import shutil
#
# drive.mount('/content/drive')
#
# save_dir = Path("/content/drive/MyDrive/road-surface-results")
# save_dir.mkdir(parents=True, exist_ok=True)
#
# # Чекпоинты
# for ckpt in glob.glob("checkpoints/*.pt"):
#     shutil.copy(ckpt, save_dir)
#     print(f"  ✓ {ckpt}")
#
# # Предсказания
# if os.path.exists("predictions.csv"):
#     shutil.copy("predictions.csv", save_dir)
#     print("  ✓ predictions.csv")
#
# # Логи
# for log in glob.glob("runs/*/metrics.jsonl"):
#     shutil.copy(log, save_dir)
#     print(f"  ✓ {log}")
#
# print(f"\n✓ Результаты сохранены в {save_dir}")

## Сравнение нескольких моделей (опционально)

Раскомментируйте ячейку ниже, чтобы обучить и сравнить несколько моделей.

In [ ]:
# # Раскомментируйте для сравнения моделей:
#
# MODELS = [
#     "configs/audio/models/simple_cnn.yaml",
#     "configs/audio/models/resnet18_mel.yaml",
#     "configs/audio/models/efficientnet_b0.yaml",
# ]
#
# for config in MODELS:
#     model_name = Path(config).stem
#     print(f"\n{'='*60}")
#     print(f"Обучение: {model_name}")
#     print(f"{'='*60}")
#
#     !python scripts/train.py \
#         --config {config} \
#         --override training.epochs=30 \
#         --logger file